In [ ]:
import numpy as np
from scipy.interpolate import LinearNDInterpolator

def load_popescu_file(filename):
    """
    Lê u_*.dat ignorando o cabeçalho e retorna:
      - pontos (R,z)
      - urad
      - eabs
    """

    # Abrir arquivo e encontrar a linha da tabela
    with open(filename, "r") as f:
        lines = f.readlines()

    # Procurar onde começa a tabela
    for i, line in enumerate(lines):
        if line.strip().startswith("r(pc)"):
            start = i + 1
            break

    # Ler o resto como tabela numpy
    data = np.genfromtxt(lines[start:], dtype=float)

    R = data[:, 0]      # pc
    Z = data[:, 1]      # pc
    urad = data[:, 2]   # erg/pc^3/Å
    eabs = data[:, 3]   # erg/s/pc^3/Å

    pts = np.column_stack([R, Z])

    interp_urad = LinearNDInterpolator(pts, urad)
    interp_eabs = LinearNDInterpolator(pts, eabs)

    return interp_urad, interp_eabs



In [ ]:
import glob
import os

class PopescuISRF:
    def __init__(self, folder):
        self.fields = {}  # { 'u_b': {'urad': interp, 'eabs': interp}, ... }

        for f in glob.glob(os.path.join(folder, "u_*.dat")):
            band = os.path.basename(f).replace(".dat", "")   # "u_b", "u_uv36"
            interp_urad, interp_eabs = load_popescu_file(f)

            self.fields[band] = {
                "urad": interp_urad,
                "eabs": interp_eabs,
            }


In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

R_SUN_PC = 8500.0  # pc

def skycoord_to_Rz(coord: SkyCoord, distance):
    """
    Retorna (R,z) em pc.
    """
    g = coord.galactic
    d = distance.to(u.pc).value

    x = R_SUN_PC - d * np.cos(g.b.radian) * np.cos(g.l.radian)
    y =          - d * np.cos(g.b.radian) * np.sin(g.l.radian)
    z =            d * np.sin(g.b.radian)

    R = np.sqrt(x*x + y*y)

    return R, z


In [ ]:
class PopescuISRF(PopescuISRF):
    def query(self, coord: SkyCoord, distance):
        R, Z = skycoord_to_Rz(coord, distance)

        result = {}

        for band, comps in self.fields.items():
            uval = comps["urad"](R, Z)
            eval = comps["eabs"](R, Z)

            result[band] = {
                "urad": float(uval) if uval is not None else np.nan,
                "eabs": float(eval) if eval is not None else np.nan,
                "temperature": None  # catálogo não fornece T
            }

        return result


In [ ]:
folder = "/home/phoenix/Coding/GitHub/feupy/feupy/naima/models"

isrf = PopescuISRF(folder)

pos = SkyCoord("18h26m00s", "-13d40m00s", frame="icrs")
dist = 4000*u.pc

vals = isrf.query(pos, dist)

print(vals["u_b"])


In [ ]:
vals

In [ ]:
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

# ---------------------
# Conversão (SkyCoord → R,z)
# ---------------------
R_SUN_PC = 8500.0  # pc

def skycoord_to_Rz(coord: SkyCoord, distance):
    g = coord.galactic
    d = distance.to(u.pc).value

    x = R_SUN_PC - d * np.cos(g.b.radian) * np.cos(g.l.radian)
    y =          - d * np.cos(g.b.radian) * np.sin(g.l.radian)
    z =            d * np.sin(g.b.radian)

    R = np.sqrt(x*x + y*y)
    return R, z


# ---------------------
# Classe principal
# ---------------------
class PopescuISRF:
    def __init__(self, folder):
        self.fields = {}  # irá conter os interpoladores
        self.load_all(folder)

    # ----- leitor automático -----
    def load_all(self, folder):
        import glob, os
        from scipy.interpolate import LinearNDInterpolator

        for path in glob.glob(os.path.join(folder, "u_*.dat")):
            band = os.path.basename(path).replace(".dat", "")

            with open(path, "r") as f:
                lines = f.readlines()

            # achar início da tabela
            for i, L in enumerate(lines):
                if L.strip().startswith("r(pc)"):
                    start = i + 1
                    break

            data = np.genfromtxt(lines[start:], dtype=float)
            R, Z = data[:,0], data[:,1]
            urad = data[:,2]

            pts = np.column_stack([R, Z])
            interp = LinearNDInterpolator(pts, urad)

            self.fields[band] = interp

    # ---------------------
    # Cálculo da densidade total U por campo (FIR, CMB, NIR...)
    # ---------------------
    def query_fields(self, coord, distance, fields=["CMB", "FIR"]):

        R, Z = skycoord_to_Rz(coord, distance)

        # grupos de bandas
        group_bands = {
            "UV":  ["u_uv09", "u_uv13", "u_uv15", "u_uv16", "u_uv20",
                    "u_uv22", "u_uv25", "u_uv28"],
            "OPT": ["u_b", "u_v", "u_uv36"],
            "NIR": ["u_i", "u_j", "u_k", "u_nir"],
            "FIR": ["u_nir"],      # substitua depois pelo grid_dust.save
        }

        # temperaturas típicas
        temps = {
            "CMB": 2.725,
            "UV": 20000,
            "OPT": 6000,
            "NIR": 2500,
            "FIR": 25,
        }

        result = {}

        for field in fields:
            if field == "CMB":
                # constante física
                u_cmb = 0.26 * 1.602e-12  # erg/cm3
                result["CMB"] = {"T": temps["CMB"], "u": u_cmb}
                continue

            if field not in group_bands:
                raise ValueError(f"Campo desconhecido: {field}")

            # soma das bandas Popescu correspondentes
            u_total = 0.0
            for band in group_bands[field]:
                if band not in self.fields:
                    continue
                val = self.fields[band](R, Z)
                if val is not None:
                    u_total += val

            result[field] = {
                "T": temps[field],
                "u": float(u_total)
            }

        return result


In [ ]:
coord = SkyCoord("18h28m00s", "-13d45m00s", frame="icrs")
dist = 4000 * u.pc

isrf = PopescuISRF("//home/phoenix/Coding/GitHub/feupy/feupy/naima/models")

res = isrf.query_fields(coord, dist, fields=["CMB", "FIR"])
print(res)


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Optional
import numpy as np
import astropy.units as u

@dataclass
class NaimaMCMCConfig:
    """
    Configuration object for running Naima MCMC analysis.
    """

    # --- Data table ---
    data_table: object = None
    data_tables: object = None  # multi-instrument

    # --- Energy range ---
    energy_bounds: tuple = (1 * u.GeV, 100 * u.TeV)

    # --- Model selection ---
    spectral_model_name: str = "PL"
    particle_model: str = "electrons"

    # --- Initial guess ---
    p0: np.ndarray | None = None

    # --- Output ---
    output_root: str = "naima_output"
    output_dir: Path = Path("./naima_results")

    # --- Sampler configuration ---
    nwalkers: int = 32
    nrun: int = 2000
    nburn: int = 500
    threads: int = 1

    # --- Priors ---
    prior_function: Optional[Callable] = None

    def ensure_paths(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
    nwalkers=48,
    nburn=100,
    nrun=500,
    threads=4,
    prefit=True,
    interactive=False

In [6]:
# Licensed under a 3-clause BSD style license - see LICENSE.rst
"""Configuration for Naima spectral modeling and MCMC analysis."""

import json
import yaml
from pathlib import Path
from gammapy.utils.scripts import make_path, to_yaml, read_yaml
from gammapy.analysis.config import GammapyBaseConfig, deep_update, GeneralConfig
from typing import List, Union
from gammapy.utils.types import EnergyType
import astropy.units as u


from pydantic import Field
from dataclasses import dataclass
from astropy import units as u
from feupy.utils.types import ParticleType

class NaimaPriorConfig(GammapyBaseConfig):
    """Prior configuration for Naima MCMC."""
    type: str = "uniform"   # uniform, loguniform, gaussian
    limits: list[float] | None = None


class NaimaSamplerConfig(GammapyBaseConfig):
    """Sampler configuration."""
    nwalkers: int = 32
    nburn: int = 2000
    nrun: int = 300
    threads: int = 1
    prefit: bool = True
    interactive: bool = False

class NaimaOutputConfig(GammapyBaseConfig):
    """Output directory and filenames."""
    root: str = "naima_output"
    overwrite: bool = True

class ICSeedFieldConfig(GammapyBaseConfig):
    name: str
    temperature: float = Field(..., description="Blackbody temperature in Kelvin")
    energy_density: float = Field(..., description="u_ph in eV/cm^3")

class PionDecayDensityConfig(GammapyBaseConfig):
    nh: float = 1.0  # in cm^-3

class ParticleEnergyConfig(GammapyBaseConfig):
    """Energy bounds for particle distributions."""
    min: EnergyType = None
    max: EnergyType = None
    ref: EnergyType = None

class NaimaModelConfig(GammapyBaseConfig):

    # particle content
    particle_type: str = "electrons"  
    # choices: "electrons", "protons", "electrons+protons"

    distribution_type: str = "ECPL"  
    # choices: "ECPL", "BPL", "LogParabola"

    # separate energy configurations
    electron_energy: ParticleEnergyConfig = ParticleEnergyConfig(
        min=1e9*u.eV,   # 1 GeV
        max=1e15*u.eV,  # 1 PeV
        ref=1e12*u.eV,  # 1 TeV
    )

    proton_energy: ParticleEnergyConfig = ParticleEnergyConfig(
        min=1e8*u.eV,    # 100 MeV (protons can go lower)
        max=1e16*u.eV,   # 10 PeV
        ref=1e12*u.eV,   # 1 TeV
    )

    # photon fields (Inverse Compton)
    seed_photon_fields: List[ICSeedFieldConfig] = [
        ICSeedFieldConfig(name="CMB", temperature=2.7, energy_density=0.25),
        ICSeedFieldConfig(name="FIR", temperature=40, energy_density=1.0),
        ICSeedFieldConfig(name="NIR", temperature=500, energy_density=0.4),
        ICSeedFieldConfig(name="VIS", temperature=3500, energy_density=1.9),
    ]

    # pion-decay target gas density
    pion_decay: PionDecayDensityConfig = PionDecayDensityConfig(nh=1.0)

    @property
    def ic_fields_naima(self):
        """Return list of IC fields formatted for Naima."""
        return [f.to_naima() for f in self.seed_photon_fields]

    def get_energy_config(self, particle):
        """
        Returns the appropriate energy config based on particle type.
        """
        if particle == "electrons":
            return self.electron_energy
        if particle == "protons":
            return self.proton_energy
        raise ValueError(f"Unknown particle type: {particle}")


class NaimaMCMCConfig(GammapyBaseConfig):
    """Main configuration class for Naima MCMC runs.
    """

    general: GeneralConfig = GeneralConfig()
    model: NaimaModelConfig = NaimaModelConfig()

    def __str__(self):
        """Display settings in pretty YAML format."""
        info = self.__class__.__name__ + "\n\n\t"
        data = self.to_yaml()
        data = data.replace("\n", "\n\t")
        info += data
        return info.expandtabs(tabsize=4)

    @classmethod
    def read(cls, path):
        """Read from YAML file.

        Parameters
        ----------
        path : str
            input filepath
        """
        config = read_yaml(path)
        config.pop("metadata", None)
        return AnalysisConfig(**config)

    @classmethod
    def from_yaml(cls, config_str):
        """Create from YAML string.

        Parameters
        ----------
        config_str : str
            yaml str
        """
        settings = yaml.safe_load(config_str)
        return AnalysisConfig(**settings)

    def write(self, path, overwrite=False):
        """Write to YAML file.

        Parameters
        ----------
        path : `pathlib.Path` or str
            Path to write files.
        overwrite : bool, optional
            Overwrite existing file. Default is False.
        """
        yaml_str = self.to_yaml()
        write_yaml(yaml_str, path, overwrite=overwrite)

    def to_yaml(self):
        """Convert to YAML string."""
        data = json.loads(self.model_dump_json())
        return to_yaml(data)

    def set_logging(self):
        """Set logging config.

        Calls ``logging.basicConfig``, i.e. adjusts global logging state.
        """
        self.general.log.level = self.general.log.level.upper()
        logging.basicConfig(**self.general.log.model_dump())
        log.info("Setting logging config: {!r}".format(self.general.log.model_dump()))

    def update(self, config=None):
        """Update config with provided settings.

        Parameters
        ----------
        config : str or `AnalysisConfig` object, optional
            Configuration settings provided in dict() syntax. Default is None.
        """
        if isinstance(config, str):
            other = AnalysisConfig.from_yaml(config)
        elif isinstance(config, AnalysisConfig):
            other = config
        else:
            raise TypeError(f"Invalid type: {config}")

        config_new = deep_update(
            self.model_dump(exclude_defaults=True),
            other.model_dump(exclude_defaults=True),
        )
        return AnalysisConfig(**config_new)

    @staticmethod
    def _get_doc_sections():
        """Return dictionary with commented docs from docs file."""
        doc = defaultdict(str)
        with open(DOCS_FILE) as f:
            for line in filter(lambda line: not line.startswith("---"), f):
                line = line.strip("\n")
                if line.startswith("# Section: "):
                    keyword = line.replace("# Section: ", "")
                doc[keyword] += line + "\n"
        return doc

In [ ]:
from naima.models import PowerLaw, 

NaimaMCMCConfig(general=GeneralConfig(log=LogConfig(level='info', filename=None, filemode=None, format=None, datefmt=None), outdir='.', n_jobs=1, datasets_file=None, models_file=None), run=NaimaRunConfig(data_table=None), model=NaimaModelConfig(particle_type='electrons', distribution_type='ECPL', electron_energy=ParticleEnergyConfig(min=None, max=None, ref=None), proton_energy=ParticleEnergyConfig(min=None, max=None, ref=None), seed_photon_fields=[ICSeedFieldConfig(name=None, temperature=None, energy_density=None), ICSeedFieldConfig(name=None, temperature=None, energy_density=None), ICSeedFieldConfig(name=None, temperature=None, energy_density=None), ICSeedFieldConfig(name=None, temperature=None, energy_density=None)], pion_decay=PionDecayDensityConfig(nh=1.0)), sampler=NaimaSamplerConfig(nwalkers=32, nburn=2000, nrun=300, threads=1, prefit=True, interactive=False), output=NaimaOutputConfig(root='naima_output', overwrite=True))

In [14]:
config

NaimaMCMCConfig(general=GeneralConfig(log=LogConfig(level='info', filename=None, filemode=None, format=None, datefmt=None), outdir='.', n_jobs=1, datasets_file=None, models_file=None), model=NaimaModelConfig(particle_type='electrons', distribution_type='ECPL', electron_energy=ParticleEnergyConfig(min=<Quantity 1.e+09 eV>, max=<Quantity 1.e+15 eV>, ref=<Quantity 1.e+12 eV>), proton_energy=ParticleEnergyConfig(min=<Quantity 1.e+08 eV>, max=<Quantity 1.e+16 eV>, ref=<Quantity 1.e+12 eV>), seed_photon_fields=[ICSeedFieldConfig(name='CMB', temperature=2.7, energy_density=0.25), ICSeedFieldConfig(name='FIR', temperature=40.0, energy_density=1.0), ICSeedFieldConfig(name='NIR', temperature=500.0, energy_density=0.4), ICSeedFieldConfig(name='VIS', temperature=3500.0, energy_density=1.9)], pion_decay=PionDecayDensityConfig(nh=1.0)), sampler=NaimaSamplerConfig(nwalkers=32, nburn=2000, nrun=300, threads=1, prefit=True, interactive=False), output=NaimaOutputConfig(root='naima_output', overwrite=True))